# Companies House - Logistic Regression

This script concentrates on creating and running the logistic regression model using data created and saved in the previous script.
We saved the cleaned and prepared dataset earlier as a Parquet file in a previous script and saved it loacally for use in this and other scripts to enable faster, more secure loading as the data is **~5.5million rows**.

This script loads the data into a DataFrame and uses it to create a logistic regression model that can for predicting the probability that a company is likely to file late Companies House accounts when considering their company structure and governance features and comparable to the late filing history of other companies with similar features.

## 1. Libraries and Data Loading

In [92]:
# Core libraries
import pandas as pd
import numpy as np

# Modelling preprocessing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# Evaluation
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    PrecisionRecallDisplay,
    classification_report,
    roc_auc_score,
    RocCurveDisplay
)
import matplotlib.pyplot as plt
import seaborn as sns

#### Load the final modelling dataset from its local file storage path.

In [93]:
# Load the final modelling dataset
data_path = r"C:\Users\AT355573\Documents\DS_Assignments\Companies House\data_processed\model_ready_data.parquet"
df = pd.read_parquet(data_path)

df.head(10)

,company_category,company_age_when_acc_due,industry,registered_country,psc_count,has_corporate_psc,has_foreign_psc,recent_psc_change,overdue
0,LTD,10–20,Public & Non‑Market,KNOWN UK,2,0,1,0,0
1,LTD,5–10,Information & Communication,UNKNOWN,1,0,0,0,0
2,LTD,0–2,Health & Social Care,KNOWN UK,1,0,0,1,0
3,LTD,2–5,"Professional, Scientific & Tech",KNOWN UK,1,0,1,0,0
4,LTD,5–10,Information & Communication,KNOWN UK,1,0,0,0,0
5,LTD,5–10,Information & Communication,KNOWN UK,1,0,1,0,0
6,LTD,5–10,Real Estate,KNOWN UK,2,0,1,0,0
7,LTD,10–20,"Professional, Scientific & Tech",KNOWN UK,3,0,0,0,1
8,LTD,10–20,"Professional, Scientific & Tech",UNKNOWN,2,0,0,0,0
9,LTD,20+,Other Services,UNKNOWN,1,0,0,0,1


In [94]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5506694 entries, 0 to 5506693
Data columns (total 9 columns):
 #   Column                    Dtype   
---  ------                    -----   
 0   company_category          category
 1   company_age_when_acc_due  category
 2   industry                  category
 3   registered_country        category
 4   psc_count                 int64   
 5   has_corporate_psc         int8    
 6   has_foreign_psc           int8    
 7   recent_psc_change         int8    
 8   overdue                   int8    
dtypes: category(4), int64(1), int8(4)
memory usage: 84.0 MB


## 2. Define Feature and Target Variables
Here our target variable is *"overdue"* which describes when a company has filed on-time (0) or is overdue (1).

In [95]:
# Separate the features (X) and target (y)

# Define the target variable, 'overdue' is the variable we want to predict (0 = on time, 1 = late).
target = "overdue"

# define X which contains all predictor/feature variables.
x = df.drop(columns=[target])

# define y which contains only the target variable
y = df[target]

Identify which columns are categorical and which are numeric as that information will be needed later for one-hot encoding.

In [96]:
# Identify categorical and numeric columns.
categorical_cols = x.select_dtypes(include="category").columns.tolist()
numeric_cols = x.select_dtypes(include=["int64", "int8", "float64"]).columns.tolist()
print("Categorical Columns: ", categorical_cols)
print("Numerical Columns:   ", numeric_cols)

Categorical Columns:  ['company_category', 'company_age_when_acc_due', 'industry', 'registered_country']
Numerical Columns:    ['psc_count', 'has_corporate_psc', 'has_foreign_psc', 'recent_psc_change']


## 3. Train-Test Split
Split the data into two sectors, one to be used as a training dataset and one to be used as a testing dataset. The training of the model should be done on unseen data so splitting the data is done before scaling and one-hot encoding to reduce the data leakage risk.
- **Scaling:** The mean and standard deviation used in *StandardScaler* would be computed on the entire dataset, which would leak information from the test set into the training process.
- **One-hot Encoding:** If a categorical variable has a rare category that only exists in the test set then doing one-hot encoding before the splitting could intriduce categories into the training that wouldn't otherwise. Or conversely, the model could see some categories that only exist in the test dataset during training.

We stratify the data within the split on the target variable to keep the **proportion** of overdue companies the same in both sets, and use a seed to ensure the results are replicable when running the process again using the same dataset.

Due to the large starting dataset size *~5.5m* a stratified sample will be taken to ensure computational efficiency while preserving the original class distribution of late filings **(8.2%)**. This approach enables efficient application of SMOTE and logistic regression without compromising the distribution of the data. We will use a strafied sample size of **~20%** of the population size.<br>
**NOTE:** The full sample should be used in a production model as maximum predictive accuracy is required and there is likely a much higher computational power available.

In [97]:
# We split the data so we can test the model on unseen data.
# test_size decides the proportion retained in the test dataset
# stratify defines which column should be used to stratify the data, y is chosen to keep the **proportion** of overdue companies the same in both sets.
# random_state acts as a seed to ensure results are replicable when running the code again with the same data.

x_train, x_test, y_train, y_test = train_test_split(
    x, y, 
    test_size = 0.2,
    stratify = y,
    random_state = 42
)

# ---------------------------------------------------------
# 5a. Take a stratified sample of the training data
# ---------------------------------------------------------
# This reduces the dataset size so SMOTE + logistic regression run much faster.
# 20% is a good balance between speed and performance.

X_train_sampled, _, y_train_sampled, _ = train_test_split(
    X_train, y_train,
    train_size=0.2,
    stratify=y_train,
    random_state=42
)

print("Training size BEFORE sampling:", len(y_train))
print("Training size AFTER sampling:", len(y_train_sampled))

# Value counts (absolute frequencies)
value_counts = y_train.value_counts()

# Percentages
percentages = y_train.value_counts(normalize=True) * 100

# Combine into a single DataFrame
df_ytrain_stats = pd.DataFrame({
    'count': value_counts,
    'percentage': percentages
})

# Optional: Reset index for cleaner column names
df_ytrain_stats = df_ytrain_stats.reset_index()
df_ytrain_stats.columns = ['value', 'count', 'percentage']

# Format the table for clean display
df_ytrain_stats.style.hide(axis="index")

value,count,percentage
0,4043709,91.790764
1,361646,8.209236


This demonstrates a highly imbalanced dataset, with only 8.2% of companies filing late. <br>
To address the imbalance, **Synthetic Minority Oversampling Technique (SMOTE)** will be applied when building the model pipeline ***(in section 5)***, synthesising new minority-class examples so that the number of examples matches the number of examples of the majority class. This will ensure that the logisitc regression model receives equal representation for both classes **during training**.
**Notes:**
- SMOTE is very resource intensive when datasets contain a large number of categories and large number of observations, like we see in the Companies House data we are using which is millions of rows.
- Starting population **~5.5**million, **80%** training data **~4.4**million, **20%** testing data **~1.1**million

## 4. Preprocessing (Encoding categorical variables and Scaling numeric variables)
Logistic regression cannot work directly with text categories, *OneHotEncoder* turns variable containing categorical data into multiple binary columns which can then be used in the logistic regression model.
<br>**NOTE:** Using *drop="first"* avoids multicollinearity as otherwise all of the one-hot encoded columns together will be able to directly predict the original column.

*StandardScaler* ensures each feature has a mean of 0 and a standard deviation of 1. This prevents features with larger numeric ranges from dominating model training in logistic regression.

In [98]:
# Use OneHotEncoder on categorical features and StandardScaler on numeric features.

preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), categorical_cols)    # drop="first" avoids multicollinearity.
    ]
)

## 5. Build Model Pipeline
The pipeling will keep the preprocessing and modelling together, to ensure that the same steps are applied consistently during training and testing, and it avoids having to manually do the transformations twice.

In [ ]:

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("logreg", LogisticRegression(max_iter=1000))
    ]
)


In [ ]:
# ---------------------------------------------------------
# 5b. Show class balance BEFORE and AFTER SMOTE (conceptual)
# ---------------------------------------------------------
print("\nClass balance BEFORE SMOTE:")
print(y_train_sampled.value_counts())

majority = y_train_sampled.value_counts()[0]

print("\nClass balance AFTER SMOTE (conceptual):")
print(pd.Series([0]*majority + [1]*majority).value_counts())


In [ ]:
# ---------------------------------------------------------
# 6. Build the modelling pipeline WITH SMOTE inside it
# ---------------------------------------------------------
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

model = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("smote", SMOTE(random_state=42)),
        ("logreg", LogisticRegression(max_iter=1000))
    ]
)


In [ ]:
# ---------------------------------------------------------
# 6. Fit the model
# ---------------------------------------------------------
# The model learns the relationship between the features and the target.

model.fit(X_train_sampled, y_train_sampled)


In [ ]:
# ---------------------------------------------------------
# GRAPH 3: Precision–Recall Curve
# ---------------------------------------------------------
# Useful when the target class is imbalanced.

PrecisionRecallDisplay.from_estimator(model, X_test, y_test)
plt.title("Precision–Recall Curve")
plt.show()

In [ ]:
# ---------------------------------------------------------
# 6a. Threshold tuning: change precision and recall
# ---------------------------------------------------------

# 1. Get predicted probabilities for class 1 (late filers)
y_scores = model.predict_proba(X_test)[:, 1]

# 2. Choose a threshold (example: 0.7)
threshold = 0.7

# 3. Evaluate at threshold 0.5
y_pred_default = (y_scores >= 0.5).astype(int)
print(classification_report(y_test, y_pred_default))


In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix
import pandas as pd

thresholds = [0.60, 0.65, 0.70, 0.75]
results = []

for t in thresholds:
    y_pred_t = (y_scores >= t).astype(int)

    prec = precision_score(y_test, y_pred_t)
    rec = recall_score(y_test, y_pred_t)
    f1 = f1_score(y_test, y_pred_t)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_t).ravel()

    results.append({
        "threshold": t,
        "precision": prec,
        "recall": rec,
        "f1": f1,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn
    })

threshold_df = pd.DataFrame(results)
threshold_df


Threshold tuning was performed to evaluate the trade‑off between precision and recall. Lower thresholds (e.g., 0.60) produced high recall but very low precision, resulting in a large number of false positives. Higher thresholds (e.g., 0.75) achieved much higher precision but at the cost of missing the majority of late filers. A threshold of 0.70 provided the most balanced performance, improving precision to 0.27 while maintaining a usable recall of 0.18 and substantially reducing false positives. This threshold aligns with the elbow of the precision–recall curve and represents the most appropriate operating point for this model.

In [ ]:
import matplotlib.pyplot as plt

thresholds = threshold_df["threshold"]
precisions = threshold_df["precision"]
recalls = threshold_df["recall"]
f1s = threshold_df["f1"]

plt.figure(figsize=(10, 6))

plt.plot(thresholds, precisions, marker='o', label='Precision', linewidth=2)
plt.plot(thresholds, recalls, marker='o', label='Recall', linewidth=2)
plt.plot(thresholds, f1s, marker='o', label='F1 Score', linewidth=2)

# Optional: highlight your chosen threshold (e.g., 0.70)
chosen = 0.70
plt.axvline(chosen, color='grey', linestyle='--', alpha=0.7)
plt.text(chosen + 0.002, 0.05, f"Chosen threshold = {chosen}", rotation=90)

plt.title("Precision, Recall, and F1 vs Threshold")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.ylim(0, 1)
plt.grid(True, linestyle='--', alpha=0.5)
plt.legend()

plt.show()


A threshold of 0.70 was selected as the final operating point. This threshold provided the best balance between precision and recall, improving precision to 0.27 while maintaining a usable recall of 0.18. It also substantially reduced false positives compared to lower thresholds and aligned with the elbow of the precision–recall curve, making it the most appropriate choice for this imbalanced classification problem

### Threshold Tuning
Logistic regression outputs a probability score between 0 and 1, representing the model’s estimated likelihood that a company will file its accounts late. The default decision threshold of 0.50 assumes equal misclassification costs and balanced classes. However, this assumption does not hold in this context: late filing is a minority class, and the dataset is highly imbalanced. As a result, using the default threshold leads to very high recall but extremely low precision for the positive class, producing an excessive number of false positives.

To address this, threshold tuning was performed to explore the trade‑off between precision and recall across a range of probability cut‑offs. Four thresholds were evaluated: 0.60, 0.65, 0.70, and 0.75. For each threshold, precision, recall, F1‑score, and confusion‑matrix counts were calculated using the model’s predicted probabilities on the test set.

The results showed a clear pattern. Lower thresholds (e.g., 0.60) produced high recall (0.51) but very low precision (0.17), meaning the model identified many late filers but at the cost of a large number of false alarms (222,989 false positives). Increasing the threshold improved precision but reduced recall. At 0.75, precision rose to 0.44, but recall fell to 0.10, indicating that the model only flagged cases where it was extremely confident, missing the majority of late filers.

A threshold of 0.70 provided the most balanced performance. At this level, precision improved to 0.27, recall remained at a usable 0.18, and false positives dropped substantially to 45,548. This threshold also aligned with the “elbow” of the precision–recall curve, where gains in precision begin to incur disproportionately large losses in recall. Selecting 0.70 therefore represents a pragmatic compromise between identifying late filers and maintaining an acceptable false‑positive rate.

Overall, threshold tuning demonstrated that the model’s practical utility depends heavily on selecting an appropriate operating point. The chosen threshold of 0.70 offers a defensible balance between precision and recall and reflects the performance characteristics most suitable for this imbalanced classification problem.




In [ ]:
# 3. Convert probabilities to class predictions
y_pred_thresh = (y_scores >= threshold).astype(int)

# 4. Evaluate performance at this threshold
from sklearn.metrics import classification_report, confusion_matrix

print(f"Metrics at threshold = {threshold}")
print(classification_report(y_test, y_pred_thresh))
print(confusion_matrix(y_test, y_pred_thresh))


In [ ]:
# ---------------------------------------------------------
# 7. Evaluate the model
# ---------------------------------------------------------
# We check how well the model performs on the test set.
# classification_report gives precision, recall, F1-score.
# AUC measures how well the model separates late vs on-time filers.

y_prob = model.predict_proba(X_test)[:, 1]

print("AUC:", roc_auc_score(y_test, y_prob))

threshold = 0.70
y_pred_thresh = (y_prob >= threshold).astype(int)

print(f"Metrics at threshold = {threshold}")
print(classification_report(y_test, y_pred_thresh))
print(confusion_matrix(y_test, y_pred_thresh))



This model achieved an accuracy of 0.92 (92%). This metric is inflated due to class imblaance in the dataset as an large majority of companies file on time. The model performs well for the majority class (on-time filers) with a precision of 0.92 and recall of 1.00, however recall for the minority class (late filers) is low at 0.03. This indicates that the model rarely predicts the minority class. Tha AUC score of 0.73 provides a reliable measure of performance which shows that the model has moderate discriminatory ability and can meaninfully rank companies by their risk of filing late, even though the classification of the late filers is limited.

In [ ]:
# ---------------------------------------------------------
# GRAPH 1: ROC Curve
# ---------------------------------------------------------
# Shows how well the model separates late vs on-time filers.

RocCurveDisplay.from_estimator(model, X_test, y_test)
plt.title("ROC Curve")
plt.show()


In [ ]:
# ---------------------------------------------------------
# GRAPH 2: Confusion Matrix
# ---------------------------------------------------------
# Shows correct and incorrect classifications.

ConfusionMatrixDisplay.from_estimator(model, X_test, y_test)
plt.title("Confusion Matrix")
plt.show()


In [ ]:
# ---------------------------------------------------------
# GRAPH 3: Precision–Recall Curve
# ---------------------------------------------------------
# Useful when the target class is imbalanced.

PrecisionRecallDisplay.from_estimator(model, X_test, y_test)
plt.title("Precision–Recall Curve")
plt.show()


In [ ]:
# ---------------------------------------------------------
# GRAPH 4: Odds Ratio Plot
# ---------------------------------------------------------
# Logistic regression coefficients can be exponentiated to get odds ratios.
# OR > 1 increases odds of filing late; OR < 1 decreases odds.

feature_names = model.named_steps["preprocess"].get_feature_names_out()
coeffs = model.named_steps["logreg"].coef_[0]
odds_ratios = np.exp(coeffs)

plt.figure(figsize=(10, 8))
plt.barh(feature_names, odds_ratios)
plt.axvline(1, color="red", linestyle="--")  # OR = 1 means no effect
plt.title("Odds Ratios for Logistic Regression Features")
plt.xlabel("Odds Ratio")
plt.tight_layout()
plt.show()


Top 5 Features (those that increase the odds of late filing)
These are the features with the highest odds ratios, meaning they push the model strongly towards predicting “late”.
1. Recent PSC change
Meaning: The company recently changed who controls it.
Interpretation: Companies going through ownership changes are more disorganised and more likely to miss deadlines.
2. Has a foreign PSC
Meaning: At least one person with significant control is based abroad.
Interpretation: Cross‑border admin is slower and more complex, increasing the chance of late filing.
3. Has a corporate PSC
Meaning: Another company controls this company.
Interpretation: Corporate ownership structures add layers of admin, which can delay filings.
4. PSC count
Meaning: The company has multiple controlling individuals/entities.
Interpretation: More PSCs = more complexity = higher risk of missing deadlines.
5. Industry: Wholesale & Retail
Meaning: The company operates in retail or wholesale.
Interpretation: These sectors often have high churn, small admin teams, and operational pressure, making late filing more common.

Bottom 5 Features (those that decrease the odds of late filing)
These features have odds ratios below 1, meaning they push the model towards predicting “on time”.
1. Company category: LTD
Meaning: Standard private limited companies.
Interpretation: These companies tend to be stable and familiar with filing requirements.
2. Company category: LTD by Guarantee / CIC
Meaning: Non‑profits, charities, community interest companies.
Interpretation: These organisations often have stronger governance and compliance processes.
3. Company category: PLC & Other
Meaning: Public limited companies and similar types.
Interpretation: Heavily regulated companies rarely miss statutory deadlines.
4. Company age: 10–20 years
Meaning: Mid‑life companies.
Interpretation: They’ve been filing for years and have established routines.
5. Company age: 20+ years
Meaning: Very mature companies.
Interpretation: Long‑established firms almost never miss deadlines — they’ve had decades to get their admin processes right.

The strongest predictors of late filing relate to ownership complexity (PSC changes, foreign PSCs, corporate PSCs, and PSC count), while the strongest predictors of on‑time filing are company maturity and stable organisational structures (older companies and standard LTD/PLC categories).


In [ ]:
# ---------------------------------------------------------
# GRAPH 5: Feature Importance (Absolute Coefficients)
# ---------------------------------------------------------
# Shows which features have the strongest influence on predictions.

importance = np.abs(coeffs)

plt.figure(figsize=(10, 8))
plt.barh(feature_names, importance)
plt.title("Feature Importance (Absolute Logistic Regression Coefficients)")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()


Top 5 Most Influential Features (push the model hardest)
These features have the biggest absolute impact on the model’s predictions — they’re the strongest signals.
1. Recent PSC change
If a company recently changed who controls it, the model sees this as a red flag.
It’s the strongest signal of potential late filing.

2. Has a foreign PSC
If someone controlling the company is based abroad, the model sees this as a risk.
International admin tends to be slower and more complex.

3. Has a corporate PSC
If another company controls this one, it adds layers of admin.
The model learns that corporate ownership often means slower filings.

4. PSC count
More people or entities in control = more complexity.
The model sees this as increasing the chance of missing deadlines.

5. Industry: Wholesale & Retail
Companies in retail or wholesale are more likely to file late.
These sectors often have high churn and small admin teams.


Bottom 5 Least Influential Features (weakest signals)
These features have the smallest absolute coefficients — they barely nudge the model’s decision.
1. Company category: LTD
Standard private limited companies are very common.
The model doesn’t learn much from this — it’s a neutral baseline.

2. Company category: LTD by Guarantee / CIC
Non‑profits and community interest companies don’t show strong filing patterns.
The model sees them as low‑impact.

3. Company category: PLC & Other
Public companies are rare in your dataset.
The model doesn’t rely on this feature much.

4. Company age: 10–20 years
Mid‑life companies are stable, but not extreme.
The model sees this age group as average — not risky, not safe.

5. Company age: 20+ years
Very old companies are reliable, but the model already learns that from other features.
This age group doesn’t add much extra signal.


The most influential features in the model relate to ownership complexity and industry sector, while company type and age have weaker effects and serve more as background context than strong predictors.

### Into Production

In a production environment, the model would receive new company records and automatically apply the same preprocessing steps used during training via the scikit‑learn pipeline. The logistic regression model would then generate a probability of late filing for each company. A threshold of 0.70 would be applied to convert this probability into a binary risk flag, allowing compliance teams to prioritise companies most likely to file late. The system could be integrated into a batch ETL workflow, an API‑driven service, or a dashboard, and would be retrained periodically to reflect changes in filing behaviour